# FG-6 — Dish details and cuisine filter

Timestamp: 2026-09-12 12:59:29 +04


## Hypothesis / intent

A cuisine-selectable candidate pool can preserve uniform direct `Math.random()` selection and prevent consecutive dish-name repeats while exposing stable, accessible dish details and a safely constructed recipe-search link.


## Method

Load the dependency-free page, syntax-check its inline JavaScript with Node, execute it against a mock DOM with deterministic randomness for 1,400 clicks, and assert data assignments, select semantics, filtering reachability, non-repetition, result/photo/detail/link updates, NBSP, and fixed layout contracts. Probe the authorized recipe URL separately with curl.


In [1]:
from pathlib import Path
import json, re, subprocess, tempfile
candidates = [Path("food_generator/index.html"), Path("../index.html")]
html_path = next(path.resolve() for path in candidates if path.exists())
html = html_path.read_text()
print(f"Loaded {html_path.name}: {len(html.splitlines())} lines")

Loaded index.html: 614 lines


In [2]:
inline_js = re.findall(r"<script>([\s\S]*?)</script>", html)
assert len(inline_js) == 1
with tempfile.NamedTemporaryFile("w", suffix=".js", delete=False) as handle:
    handle.write(inline_js[0])
    inline_path = Path(handle.name)
syntax_run = subprocess.run(["node", "--check", str(inline_path)], capture_output=True, text=True)
inline_path.unlink()
assert syntax_run.returncode == 0, syntax_run.stderr
print("inline JS syntax: PASS")

inline JS syntax: PASS


In [3]:
js_checker = 'const fs = require("fs");\nconst vm = require("vm");\nconst assert = require("assert");\nconst html = fs.readFileSync(process.argv[2], "utf8");\nconst scriptMatch = html.match(/<script>([\\s\\S]*?)<\\/script>/);\nassert(scriptMatch, "inline script missing");\n\nclass Classes {\n  constructor(...names) { this.names = new Set(names); }\n  add(name) { this.names.add(name); }\n  remove(name) { this.names.delete(name); }\n  contains(name) { return this.names.has(name); }\n}\nclass Element {\n  constructor(...classes) {\n    this.classList = new Classes(...classes);\n    this.attrs = {};\n    this.listeners = {};\n    this.textContent = "";\n    this.value = "All cuisines";\n    this.src = "";\n    this.alt = "";\n    this.hidden = false;\n    this.offsetWidth = 500;\n  }\n  addEventListener(type, callback) { this.listeners[type] = callback; }\n  removeAttribute(name) { delete this.attrs[name]; }\n}\nconst ids = {\n  "#generate-button": new Element(),\n  "#cuisine-filter": new Element(),\n  "#result": new Element(),\n  "#food-photo": new Element(),\n  "#photo-placeholder": new Element(),\n  "#photo-credit": new Element(),\n  "#detail-cuisine": new Element(),\n  "#detail-time": new Element(),\n  "#detail-difficulty": new Element(),\n  "#recipe-link": new Element("recipe-link", "is-hidden")\n};\nids["#recipe-link"].attrs["aria-hidden"] = "true";\nlet state = 0x6d2b79f5;\nconst deterministicMath = Object.create(Math);\ndeterministicMath.random = () => {\n  state = (Math.imul(state, 1664525) + 1013904223) >>> 0;\n  return state / 0x100000000;\n};\nconst context = {\n  document: { querySelector: (selector) => {\n    assert(ids[selector], `unexpected selector ${selector}`);\n    return ids[selector];\n  } },\n  Math: deterministicMath,\n  encodeURIComponent\n};\nvm.createContext(context);\nvm.runInContext(scriptMatch[1] + "\\nglobalThis.__lunchOptions = lunchOptions;", context);\nconst dishes = JSON.parse(JSON.stringify(context.__lunchOptions));\nassert.strictEqual(dishes.length, 20);\nconst expected = {\n  "Pizza":["Italian","25 min","Easy"], "Sushi":["Japanese","45 min","Medium"],\n  "Tacos":["Mexican","25 min","Easy"], "Pasta":["Italian","20 min","Easy"],\n  "Burger":["American","25 min","Easy"], "Ramen":["Japanese","35 min","Medium"],\n  "Caesar Salad":["American","15 min","Easy"], "Chicken Curry":["Asian","40 min","Medium"],\n  "Pad Thai":["Asian","30 min","Medium"], "Burrito":["Mexican","25 min","Easy"],\n  "Fried Rice":["Asian","20 min","Easy"], "Pho":["Asian","60 min","Hard"],\n  "Dumplings":["Asian","50 min","Medium"], "Falafel Bowl":["Mediterranean","35 min","Medium"],\n  "Paella":["Mediterranean","50 min","Hard"], "Bibimbap":["Asian","35 min","Medium"],\n  "Club Sandwich":["American","15 min","Easy"], "Poke Bowl":["American","20 min","Easy"],\n  "Kebab":["Mediterranean","40 min","Medium"], "Grilled Cheese":["American","15 min","Easy"]\n};\nfor (const dish of dishes) {\n  assert(dish.cuisine && dish.time && dish.difficulty);\n  assert.deepStrictEqual([dish.cuisine, dish.time, dish.difficulty], expected[dish.name]);\n}\nassert.strictEqual(Object.keys(expected).length, 20);\nconst selectBody = html.match(/<select[^>]*id="cuisine-filter"[^>]*>([\\s\\S]*?)<\\/select>/)[1];\nconst options = [...selectBody.matchAll(/<option value="([^"]+)">([^<]+)<\\/option>/g)].map(m => [m[1],m[2]]);\nconst cuisineValues = ["All cuisines","American","Asian","Italian","Japanese","Mediterranean","Mexican"];\nassert.deepStrictEqual(options, cuisineValues.map(x => [x,x]));\nfor (const cuisine of cuisineValues.slice(1)) {\n  assert(dishes.filter(d => d.cuisine === cuisine).length >= 2);\n}\nassert(/<label[^>]*for="cuisine-filter"/.test(html));\nassert(/<dl class="detail-list">/.test(html));\nassert.strictEqual((html.match(/<dt>/g) || []).length, 3);\nassert(/id="recipe-link"[^>]*target="_blank"[^>]*rel="noopener noreferrer"[^>]*aria-hidden="true"/.test(html));\nassert(/\\.recipe-link\\.is-hidden\\s*{\\s*visibility:\\s*hidden;/.test(html));\nassert(/\\.dish-details\\s*{[^}]*min-height:\\s*142px;/.test(html));\nassert(!html.includes("innerHTML"));\nassert(html.includes(\'let previousDishName = null;\'));\nassert(html.includes(\'Math.floor(Math.random() * availableOptions.length)\'));\nassert(!html.includes("previousIndex"));\nassert(/\\.result\\s*{[^}]*height:\\s*2\\.5em;[^}]*min-height:\\s*58px;[^}]*overflow:\\s*hidden;/.test(html));\nassert(/\\.food-photo\\s*{[^}]*position:\\s*absolute;[^}]*inset:\\s*0;[^}]*width:\\s*100%;[^}]*height:\\s*100%;[^}]*object-fit:\\s*cover;[^}]*object-position:\\s*center;/.test(html));\n\nconst click = ids["#generate-button"].listeners.click;\nassert.strictEqual(typeof click, "function");\nlet previous = null;\nlet clicks = 0;\nfor (const cuisine of cuisineValues) {\n  ids["#cuisine-filter"].value = cuisine;\n  const pool = cuisine === "All cuisines" ? dishes : dishes.filter(d => d.cuisine === cuisine);\n  const seen = new Set();\n  for (let i = 0; i < 200; i++) {\n    click();\n    clicks++;\n    const match = ids["#result"].textContent.match(/^Today\'s pick: (.+)!\\u00a0🥳$/u);\n    assert(match, `bad result ${ids["#result"].textContent}`);\n    const chosen = dishes.find(d => d.name === match[1]);\n    assert(chosen);\n    assert(pool.some(d => d.name === chosen.name));\n    assert.notStrictEqual(chosen.name, previous);\n    previous = chosen.name;\n    seen.add(chosen.name);\n    const emojiIndex = ids["#result"].textContent.indexOf("🥳");\n    assert.strictEqual(ids["#result"].textContent.charCodeAt(emojiIndex - 1), 160);\n    assert.strictEqual(ids["#food-photo"].src, chosen.photo);\n    assert.strictEqual(ids["#food-photo"].alt, `A plate of ${chosen.name}`);\n    assert.strictEqual(ids["#detail-cuisine"].textContent, chosen.cuisine);\n    assert.strictEqual(ids["#detail-time"].textContent, chosen.time);\n    assert.strictEqual(ids["#detail-difficulty"].textContent, chosen.difficulty);\n    assert.strictEqual(ids["#recipe-link"].textContent, "Find a recipe");\n    assert.strictEqual(ids["#recipe-link"].href, `https://www.allrecipes.com/search?q=${encodeURIComponent(chosen.name)}`);\n    assert(!ids["#recipe-link"].classList.contains("is-hidden"));\n    assert.strictEqual(ids["#recipe-link"].attrs["aria-hidden"], undefined);\n    assert(ids["#food-photo"].classList.contains("is-visible"));\n    assert(ids["#photo-credit"].classList.contains("is-visible"));\n    assert.strictEqual(ids["#photo-placeholder"].hidden, true);\n  }\n  assert.strictEqual(seen.size, pool.length, `${cuisine}: ${seen.size}/${pool.length} reached`);\n}\nassert(clicks >= 100);\nconsole.log(JSON.stringify({dishes: dishes.length, options: options.length, clicks, cuisines: cuisineValues.length, allCandidatesReached: true, noConsecutiveRepeats: true, domDetails: true, nbsp: 160, imageContract: true, resultHeight: true}));\n'
with tempfile.NamedTemporaryFile("w", suffix=".js", delete=False) as handle:
    handle.write(js_checker)
    checker_path = Path(handle.name)
check_run = subprocess.run(["node", str(checker_path), str(html_path)], capture_output=True, text=True)
checker_path.unlink()
assert check_run.returncode == 0, check_run.stderr
summary = json.loads(check_run.stdout)
assert summary == {"dishes":20,"options":7,"clicks":1400,"cuisines":7,"allCandidatesReached":True,"noConsecutiveRepeats":True,"domDetails":True,"nbsp":160,"imageContract":True,"resultHeight":True}
print(json.dumps(summary, sort_keys=True))

{"allCandidatesReached": true, "clicks": 1400, "cuisines": 7, "dishes": 20, "domDetails": true, "imageContract": true, "nbsp": 160, "noConsecutiveRepeats": true, "options": 7, "resultHeight": true}


In [4]:
network_run = subprocess.run([
    "curl", "-L", "-sS", "-o", "/dev/null", "-w", "%{http_code}",
    "--max-time", "20", "https://www.allrecipes.com/search?q=Pizza"
], capture_output=True, text=True)
print(f"curl exit={network_run.returncode}, HTTP={network_run.stdout or 'none'}")

curl exit=0, HTTP=402


## Interpretation

The local deterministic checks passed all requested data, filtering, no-repeat, accessibility-marker, exact result-text, image-cropping, stable-dimension, and recipe-link assertions. The recipe endpoint was reachable, but returned HTTP 402 to the automated curl request; this is recorded as an external bot/access limitation and the authorized URL was not changed.


In [5]:
assert network_run.returncode == 0
assert network_run.stdout == "402"  # Reachable, but automated request was denied.
print("FG-6 replay status: local acceptance PASS; external endpoint bot-blocked with HTTP 402")

FG-6 replay status: local acceptance PASS; external endpoint bot-blocked with HTTP 402
